# Typhoon OCR 1.5 resolution sweep: raw versus gray220

Full 21-page OCR output is retained in this notebook for every run. The matrix is six longest-edge settings (`1024`, `1400`, `1600`, `1800`, `2000`, `2200`) crossed with original raw images and `gray220` images. Per-page preprocessing audit, GPU snapshots, benchmark configuration, timing, and Golden scores are captured as cell output.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import time
import urllib.request

ROOT = Path(".")
AUDIT_PYTHON = Path("python")
GOLDEN = ROOT / "results/bot_credit_bureau_21p_golden_transcript_20260829.json"
SWEEP_ROOT = ROOT / "inputs/bot_credit_bureau_2559/resolution_sweep"
MANIFEST = SWEEP_ROOT / "gray220_resolution_sweep_audit.json"
MODEL = "typhoon-ocr1.5-2b"
ENDPOINT = "http://127.0.0.1:8096"
PROMPT_PROFILE = "typhoon"
CONCURRENCY = 7
SERVER_MAX_NUM_SEQS = 7
MTP_TOKENS = 0
DISABLE_THINKING = True
SIZES = (1024, 1400, 1600, 1800, 2000, 2200)
TEMPERATURE = 0.0
TOP_P = 0.8
TOP_K = 20
REPETITION_PENALTY = 1.05
PRESENCE_PENALTY = 0.0
MAX_TOKENS = 8192

def gpu_snapshot():
    return subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used", "--format=csv,noheader,nounits"],
        text=True,
    ).strip()

def wait_for_model(timeout_seconds=900):
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        try:
            with urllib.request.urlopen(ENDPOINT + "/v1/models", timeout=5) as response:
                payload = json.load(response)
            print("Endpoint ready:", ", ".join(item["id"] for item in payload.get("data", [])))
            return
        except Exception as error:
            last_error = error
            time.sleep(5)
    raise TimeoutError(f"Endpoint did not become ready: {last_error}")

def run_ocr(label, image_directory, result_path):
    output = ROOT / result_path
    if output.exists():
        raise FileExistsError(f"Refusing to overwrite {output}")
    command = [
        str(AUDIT_PYTHON), "benchmark/ocr_benchmark.py", str(ROOT / image_directory), str(output),
        "--endpoint", ENDPOINT,
        "--model", MODEL,
        "--prompt-profile", PROMPT_PROFILE,
        "--temperature", str(TEMPERATURE),
        "--top-p", str(TOP_P),
        "--top-k", str(TOP_K),
        "--repetition-penalty", str(REPETITION_PENALTY),
        "--presence-penalty", str(PRESENCE_PENALTY),
        "--max-tokens", str(MAX_TOKENS),
        "--concurrency", str(CONCURRENCY),
        "--images-per-request", "1",
        "--server-max-num-seqs", str(SERVER_MAX_NUM_SEQS),
    ]
    if DISABLE_THINKING:
        command.append("--disable-thinking")
    print(f"\n=== {label} ===")
    print("GPU before:\n" + gpu_snapshot())
    print("Command:", " ".join(command))
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=False)
    print(completed.stdout)
    if completed.stderr:
        print("STDERR:\n" + completed.stderr)
    print("GPU after:\n" + gpu_snapshot())
    if completed.returncode:
        raise RuntimeError(f"OCR exited with {completed.returncode}")
    return output

def show_all_pages(result_path):
    payload = json.loads(Path(result_path).read_text(encoding="utf-8"))
    print(json.dumps({"configuration": payload["configuration"], "summary": payload["summary"]}, ensure_ascii=False, indent=2))
    records = sorted(payload["records"], key=lambda item: int(Path(item["images"][0]).stem.split("-")[-1]))
    for record in records:
        page = int(Path(record["images"][0]).stem.split("-")[-1])
        print(f"\n--- OCR output: page {page:02d} ({record['elapsed_seconds']}s) ---")
        print(record.get("text", ""))

def show_gray_audit(longest_edge):
    payload = json.loads(MANIFEST.read_text(encoding="utf-8"))
    records = payload["sizes"][str(longest_edge)]["records"]
    print(json.dumps(records, ensure_ascii=False, indent=2))


In [2]:
sys.path.insert(0, str(ROOT / "benchmark"))
from ocr_benchmark import PROMPTS

print("Prompt profile:", PROMPT_PROFILE)
print(PROMPTS[PROMPT_PROFILE])
print("\nRun controls:")
print(json.dumps({
    "model": MODEL,
    "endpoint": ENDPOINT,
    "MTP_tokens": MTP_TOKENS,
    "server_max_num_seqs": SERVER_MAX_NUM_SEQS,
    "concurrency": CONCURRENCY,
    "disable_thinking": DISABLE_THINKING,
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
}, ensure_ascii=False, indent=2))
wait_for_model()


Prompt profile: typhoon
Extract all text from the image.

Instructions:
- Only return the clean Markdown.
- Do not include any explanation or extra text.
- You must include all information on the page.

Formatting Rules:
- Tables: Render tables using <table>...</table> in clean HTML format.
- Equations: Render equations using LaTeX syntax with inline ($...$) and block ($$...$$).
- Images/Charts/Diagrams: Wrap any clearly defined visual areas (e.g. charts, diagrams, pictures) in:
<figure>
Describe the image's main elements, visible text, and contextual clues in Thai.
</figure>
- Page Numbers: Wrap page numbers in <page_number>...</page_number>.
- Checkboxes: Use [ ] for unchecked and [x] for checked boxes.

Run controls:
{
  "model": "typhoon-ocr1.5-2b",
  "endpoint": "http://127.0.0.1:8096",
  "MTP_tokens": 0,
  "server_max_num_seqs": 7,
  "concurrency": 7,
  "disable_thinking": true,
  "temperature": 0.0,
  "max_tokens": 8192
}
Endpoint ready: typhoon-ocr1.5-2b


## 1024px raw

In [3]:
raw_1024 = run_ocr("raw 1024px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024", "results/typhoon_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json")
show_all_pages(raw_1024)



=== raw 1024px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 35649
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024 ./results/typhoon_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8096 --model typhoon-ocr1.5-2b --prompt-profile typhoon --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7 --disable-thinking


completed request 6/21 (pages 006)
completed request 3/21 (pages 003)
completed request 2/21 (pages 002)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 12/21 (pages 012)
completed request 11/21 (pages 011)
completed request 14/21 (pages 014)
completed request 16/21 (pages 016)
completed request 15/21 (pages 015)
completed request 17/21 (pages 017)
completed request 18/21 (pages 018)
completed request 19/21 (pages 019)
completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 7/21 (pages 007)
completed request 20/21 (pages 020)
completed request 21/21 (pages 021)
{"images": 21, "requests": 21, "elapsed_seconds": 78.7828, "seconds_per_image": 3.7516, "completion_tokens": 67983, "end_to_end_completion_tokens_per_second": 862.916}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 36477
{
  "c

## 1024px gray220

In [4]:
show_gray_audit(1024)
gray_1024 = run_ocr("gray220 1024px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1024", "results/typhoon_bot_credit_bureau_21p_gray220_px1024_resolution_sweep_20260829.json")
show_all_pages(gray_1024)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1024/page-01.png",
    "dimensions": [
      724,
      1024
    ],
    "threshold": 220,
    "pixels_replaced": 701722,
    "pixels_total": 741376,
    "pixels_replaced_percent": 94.6513,
    "raw_sha256": "52d5568aaef582dcec78889ffe4cf949683dd20c55c73ad0efb135690563596e",
    "gray220_sha256": "361f82d8dcccbf24b406e0de1d588675d31b4ba41964aeb8dddd18dc65ee73a3"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1024/page-02.png",
    "dimensions": [
      724,
      1024
    ],
    "threshold": 220,
    "pi

completed request 5/21 (pages 005)
completed request 1/21 (pages 001)
completed request 4/21 (pages 004)
completed request 6/21 (pages 006)
completed request 3/21 (pages 003)
completed request 7/21 (pages 007)
completed request 2/21 (pages 002)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 14/21 (pages 014)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 15/21 (pages 015)
completed request 17/21 (pages 017)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 19.0644, "seconds_per_image": 0.9078, "completion_tokens": 22434, "end_to_end_completion_tokens_per_second": 1176.745}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 36477
{
  "

## 1400px raw

In [5]:
raw_1400 = run_ocr("raw 1400px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400", "results/typhoon_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json")
show_all_pages(raw_1400)



=== raw 1400px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 36477
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400 ./results/typhoon_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8096 --model typhoon-ocr1.5-2b --prompt-profile typhoon --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7 --disable-thinking


completed request 4/21 (pages 004)
completed request 5/21 (pages 005)
completed request 6/21 (pages 006)
completed request 2/21 (pages 002)
completed request 3/21 (pages 003)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 11/21 (pages 011)
completed request 10/21 (pages 010)
completed request 12/21 (pages 012)
completed request 13/21 (pages 013)
completed request 16/21 (pages 016)
completed request 17/21 (pages 017)
completed request 15/21 (pages 015)
completed request 1/21 (pages 001)
completed request 7/21 (pages 007)
completed request 14/21 (pages 014)
completed request 18/21 (pages 018)
completed request 19/21 (pages 019)
completed request 20/21 (pages 020)
completed request 21/21 (pages 021)
{"images": 21, "requests": 21, "elapsed_seconds": 85.9755, "seconds_per_image": 4.0941, "completion_tokens": 75814, "end_to_end_completion_tokens_per_second": 881.81}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 36489
{
  "co

## 1400px gray220

In [6]:
show_gray_audit(1400)
gray_1400 = run_ocr("gray220 1400px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1400", "results/typhoon_bot_credit_bureau_21p_gray220_px1400_resolution_sweep_20260829.json")
show_all_pages(gray_1400)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1400/page-01.png",
    "dimensions": [
      990,
      1400
    ],
    "threshold": 220,
    "pixels_replaced": 1323612,
    "pixels_total": 1386000,
    "pixels_replaced_percent": 95.4987,
    "raw_sha256": "027ea1c68561df52ce2beea388ee8c612768006d6feb1f4fa09bcabcff165a18",
    "gray220_sha256": "48c9be1e4a099ded0fdef40c00daade1db540a7efd6315a691070fb821bf077c"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1400/page-02.png",
    "dimensions": [
      990,
      1400
    ],
    "threshold": 220,
    "

completed request 5/21 (pages 005)
completed request 1/21 (pages 001)
completed request 4/21 (pages 004)
completed request 6/21 (pages 006)
completed request 3/21 (pages 003)
completed request 7/21 (pages 007)
completed request 2/21 (pages 002)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 11/21 (pages 011)
completed request 14/21 (pages 014)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 15/21 (pages 015)
completed request 17/21 (pages 017)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 23.2408, "seconds_per_image": 1.1067, "completion_tokens": 22469, "end_to_end_completion_tokens_per_second": 966.791}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 36614
{
  "c

## 1600px raw

In [7]:
raw_1600 = run_ocr("raw 1600px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600", "results/typhoon_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json")
show_all_pages(raw_1600)



=== raw 1600px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 36614
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600 ./results/typhoon_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8096 --model typhoon-ocr1.5-2b --prompt-profile typhoon --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7 --disable-thinking


completed request 1/21 (pages 001)
completed request 4/21 (pages 004)
completed request 6/21 (pages 006)
completed request 5/21 (pages 005)
completed request 3/21 (pages 003)
completed request 7/21 (pages 007)
completed request 2/21 (pages 002)
completed request 8/21 (pages 008)
completed request 14/21 (pages 014)
completed request 10/21 (pages 010)
completed request 11/21 (pages 011)
completed request 13/21 (pages 013)
completed request 12/21 (pages 012)
completed request 16/21 (pages 016)
completed request 15/21 (pages 015)
completed request 19/21 (pages 019)
completed request 21/21 (pages 021)
completed request 9/21 (pages 009)
completed request 17/21 (pages 017)
completed request 18/21 (pages 018)
completed request 20/21 (pages 020)
{"images": 21, "requests": 21, "elapsed_seconds": 64.8172, "seconds_per_image": 3.0865, "completion_tokens": 54993, "end_to_end_completion_tokens_per_second": 848.433}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 36625
{
  "c

## 1600px gray220

In [8]:
show_gray_audit(1600)
gray_1600 = run_ocr("gray220 1600px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1600", "results/typhoon_bot_credit_bureau_21p_gray220_px1600_resolution_sweep_20260829.json")
show_all_pages(gray_1600)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1600/page-01.png",
    "dimensions": [
      1131,
      1600
    ],
    "threshold": 220,
    "pixels_replaced": 1732917,
    "pixels_total": 1809600,
    "pixels_replaced_percent": 95.7624,
    "raw_sha256": "7cd88e5cf12e59d83931179971c152df49e476e3e08d7ba1eb6be1fa0f5d46b7",
    "gray220_sha256": "eaf2a4fa6c8aa8ea8200b12cd7eaaa81dd4aa418aab469cf58f59cbc1d5c13e2"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1600/page-02.png",
    "dimensions": [
      1131,
      1600
    ],
    "threshold": 220,
   

completed request 5/21 (pages 005)
completed request 1/21 (pages 001)
completed request 4/21 (pages 004)
completed request 6/21 (pages 006)
completed request 3/21 (pages 003)
completed request 7/21 (pages 007)
completed request 2/21 (pages 002)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 11/21 (pages 011)
completed request 14/21 (pages 014)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 15/21 (pages 015)
completed request 20/21 (pages 020)
completed request 17/21 (pages 017)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 22.9896, "seconds_per_image": 1.0947, "completion_tokens": 22490, "end_to_end_completion_tokens_per_second": 978.267}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 36629
{
  "c

## 1800px raw

In [9]:
raw_1800 = run_ocr("raw 1800px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800", "results/typhoon_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json")
show_all_pages(raw_1800)



=== raw 1800px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 36629
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800 ./results/typhoon_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8096 --model typhoon-ocr1.5-2b --prompt-profile typhoon --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7 --disable-thinking


completed request 1/21 (pages 001)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 2/21 (pages 002)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 11/21 (pages 011)
completed request 10/21 (pages 010)
completed request 12/21 (pages 012)
completed request 13/21 (pages 013)
completed request 14/21 (pages 014)
completed request 16/21 (pages 016)
completed request 17/21 (pages 017)
completed request 5/21 (pages 005)
completed request 6/21 (pages 006)
completed request 9/21 (pages 009)
completed request 21/21 (pages 021)
completed request 15/21 (pages 015)
completed request 18/21 (pages 018)
completed request 19/21 (pages 019)
completed request 20/21 (pages 020)
{"images": 21, "requests": 21, "elapsed_seconds": 90.9794, "seconds_per_image": 4.3324, "completion_tokens": 76045, "end_to_end_completion_tokens_per_second": 835.848}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 36628
{
  "c

## 1800px gray220

In [10]:
show_gray_audit(1800)
gray_1800 = run_ocr("gray220 1800px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1800", "results/typhoon_bot_credit_bureau_21p_gray220_px1800_resolution_sweep_20260829.json")
show_all_pages(gray_1800)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1800/page-01.png",
    "dimensions": [
      1272,
      1800
    ],
    "threshold": 220,
    "pixels_replaced": 2197434,
    "pixels_total": 2289600,
    "pixels_replaced_percent": 95.9746,
    "raw_sha256": "ea45e6faa8ac8d1bc413f017d0428dae7458faee05babfde03c571c008faf256",
    "gray220_sha256": "e9aaa8c9c0ebf36bd4f65e716f6e8ebe06fa133de12a32db5f5af0df5df13234"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1800/page-02.png",
    "dimensions": [
      1272,
      1800
    ],
    "threshold": 220,
   

completed request 5/21 (pages 005)
completed request 1/21 (pages 001)
completed request 4/21 (pages 004)
completed request 6/21 (pages 006)
completed request 3/21 (pages 003)
completed request 7/21 (pages 007)
completed request 2/21 (pages 002)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 11/21 (pages 011)
completed request 14/21 (pages 014)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 15/21 (pages 015)
completed request 17/21 (pages 017)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 23.2669, "seconds_per_image": 1.1079, "completion_tokens": 22503, "end_to_end_completion_tokens_per_second": 967.17}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 36903
{
  "co

## 2000px raw

In [11]:
raw_2000 = run_ocr("raw 2000px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000", "results/typhoon_bot_credit_bureau_21p_raw_px2000_resolution_sweep_20260829.json")
show_all_pages(raw_2000)



=== raw 2000px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 36903
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000 ./results/typhoon_bot_credit_bureau_21p_raw_px2000_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8096 --model typhoon-ocr1.5-2b --prompt-profile typhoon --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7 --disable-thinking


completed request 1/21 (pages 001)
completed request 4/21 (pages 004)
completed request 5/21 (pages 005)
completed request 3/21 (pages 003)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 14/21 (pages 014)
completed request 15/21 (pages 015)
completed request 16/21 (pages 016)
completed request 17/21 (pages 017)
completed request 19/21 (pages 019)
completed request 2/21 (pages 002)
completed request 6/21 (pages 006)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 18/21 (pages 018)
completed request 20/21 (pages 020)
completed request 21/21 (pages 021)
{"images": 21, "requests": 21, "elapsed_seconds": 95.1145, "seconds_per_image": 4.5293, "completion_tokens": 76465, "end_to_end_completion_tokens_per_second": 803.926}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 36855
{
  "c

## 2000px gray220

In [12]:
show_gray_audit(2000)
gray_2000 = run_ocr("gray220 2000px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2000", "results/typhoon_bot_credit_bureau_21p_gray220_px2000_resolution_sweep_20260829.json")
show_all_pages(gray_2000)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2000/page-01.png",
    "dimensions": [
      1414,
      2000
    ],
    "threshold": 220,
    "pixels_replaced": 2718321,
    "pixels_total": 2828000,
    "pixels_replaced_percent": 96.1217,
    "raw_sha256": "d4a80b77100cb787878ee295a47b493e6c6afbe6b6619cf11bad88dc30e67a46",
    "gray220_sha256": "ee2902a15cdec3354762e8361c56bb13eb98bf8e0675d74bf0f719754b0f9ae3"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2000/page-02.png",
    "dimensions": [
      1414,
      2000
    ],
    "threshold": 220,
   

completed request 5/21 (pages 005)
completed request 1/21 (pages 001)
completed request 4/21 (pages 004)
completed request 6/21 (pages 006)
completed request 7/21 (pages 007)
completed request 3/21 (pages 003)
completed request 2/21 (pages 002)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 11/21 (pages 011)
completed request 13/21 (pages 013)
completed request 14/21 (pages 014)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 15/21 (pages 015)
completed request 17/21 (pages 017)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 24.7201, "seconds_per_image": 1.1771, "completion_tokens": 22517, "end_to_end_completion_tokens_per_second": 910.879}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 37206
{
  "c

## 2200px raw

In [13]:
raw_2200 = run_ocr("raw 2200px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200", "results/typhoon_bot_credit_bureau_21p_raw_px2200_resolution_sweep_20260829.json")
show_all_pages(raw_2200)



=== raw 2200px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 37206
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200 ./results/typhoon_bot_credit_bureau_21p_raw_px2200_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8096 --model typhoon-ocr1.5-2b --prompt-profile typhoon --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7 --disable-thinking


completed request 1/21 (pages 001)
completed request 4/21 (pages 004)
completed request 5/21 (pages 005)
completed request 3/21 (pages 003)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 12/21 (pages 012)
completed request 14/21 (pages 014)
completed request 2/21 (pages 002)
completed request 6/21 (pages 006)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 11/21 (pages 011)
completed request 16/21 (pages 016)
completed request 17/21 (pages 017)
completed request 13/21 (pages 013)
completed request 19/21 (pages 019)
completed request 15/21 (pages 015)
completed request 18/21 (pages 018)
completed request 20/21 (pages 020)
completed request 21/21 (pages 021)
{"images": 21, "requests": 21, "elapsed_seconds": 126.6634, "seconds_per_image": 6.0316, "completion_tokens": 97096, "end_to_end_completion_tokens_per_second": 766.567}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 37628
{
  "

## 2200px gray220

In [14]:
show_gray_audit(2200)
gray_2200 = run_ocr("gray220 2200px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2200", "results/typhoon_bot_credit_bureau_21p_gray220_px2200_resolution_sweep_20260829.json")
show_all_pages(gray_2200)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2200/page-01.png",
    "dimensions": [
      1555,
      2200
    ],
    "threshold": 220,
    "pixels_replaced": 3302179,
    "pixels_total": 3421000,
    "pixels_replaced_percent": 96.5267,
    "raw_sha256": "289cc186f8272818eebe04ab7817c751400dc99e89e853689e300a2a35c950b1",
    "gray220_sha256": "aca77e3bbbc84f3448de60b79c196caa439ce1ccdefeb9725d48733840f59327"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2200/page-02.png",
    "dimensions": [
      1555,
      2200
    ],
    "threshold": 220,
   

completed request 5/21 (pages 005)
completed request 1/21 (pages 001)
completed request 4/21 (pages 004)
completed request 6/21 (pages 006)
completed request 3/21 (pages 003)
completed request 7/21 (pages 007)
completed request 2/21 (pages 002)
completed request 9/21 (pages 009)
completed request 8/21 (pages 008)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 11/21 (pages 011)
completed request 14/21 (pages 014)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 15/21 (pages 015)
completed request 17/21 (pages 017)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 27.8175, "seconds_per_image": 1.3246, "completion_tokens": 22528, "end_to_end_completion_tokens_per_second": 809.851}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 37623
{
  "c

## Golden score for every size and preprocessing variant

In [15]:
score_path = ROOT / "results/typhoon_resolution_sweep_raw_gray220_20260829.score.json"
score_command = [str(AUDIT_PYTHON), "benchmark/score_golden_ocr.py", str(GOLDEN), str(score_path), "--target", "raw_px1024=results/typhoon_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json", "--target", "gray220_px1024=results/typhoon_bot_credit_bureau_21p_gray220_px1024_resolution_sweep_20260829.json", "--target", "raw_px1400=results/typhoon_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json", "--target", "gray220_px1400=results/typhoon_bot_credit_bureau_21p_gray220_px1400_resolution_sweep_20260829.json", "--target", "raw_px1600=results/typhoon_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json", "--target", "gray220_px1600=results/typhoon_bot_credit_bureau_21p_gray220_px1600_resolution_sweep_20260829.json", "--target", "raw_px1800=results/typhoon_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json", "--target", "gray220_px1800=results/typhoon_bot_credit_bureau_21p_gray220_px1800_resolution_sweep_20260829.json", "--target", "raw_px2000=results/typhoon_bot_credit_bureau_21p_raw_px2000_resolution_sweep_20260829.json", "--target", "gray220_px2000=results/typhoon_bot_credit_bureau_21p_gray220_px2000_resolution_sweep_20260829.json", "--target", "raw_px2200=results/typhoon_bot_credit_bureau_21p_raw_px2200_resolution_sweep_20260829.json", "--target", "gray220_px2200=results/typhoon_bot_credit_bureau_21p_gray220_px2200_resolution_sweep_20260829.json"]
print("Command:", " ".join(score_command))
completed = subprocess.run(score_command, cwd=ROOT, text=True, capture_output=True, check=True)
print(completed.stdout)
if completed.stderr:
    print("STDERR:\n" + completed.stderr)
score_payload = json.loads(score_path.read_text(encoding="utf-8"))
for label, target in score_payload["targets"].items():
    summary = target["summary"]
    print("\n", label)
    print(json.dumps({
        "word_accuracy_percent": summary["words"]["accuracy_percent"],
        "word_deltas": {key: summary["words"][key] for key in ("substituted", "missing", "extra")},
        "number_accuracy_percent": summary["number_tokens"]["accuracy_percent"],
    }, ensure_ascii=False, indent=2))


Command: python benchmark/score_golden_ocr.py ./results/bot_credit_bureau_21p_golden_transcript_20260829.json ./results/typhoon_resolution_sweep_raw_gray220_20260829.score.json --target raw_px1024=results/typhoon_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json --target gray220_px1024=results/typhoon_bot_credit_bureau_21p_gray220_px1024_resolution_sweep_20260829.json --target raw_px1400=results/typhoon_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json --target gray220_px1400=results/typhoon_bot_credit_bureau_21p_gray220_px1400_resolution_sweep_20260829.json --target raw_px1600=results/typhoon_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json --target gray220_px1600=results/typhoon_bot_credit_bureau_21p_gray220_px1600_resolution_sweep_20260829.json --target raw_px1800=results/typhoon_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json --target gray220_px1800=results/typhoon_bot_credit_bureau_21p_gray220_px1800_resolution_sweep_20260

{"raw_px1024": {"scored_pages": 21, "words": {"correct": 5511, "substituted": 1270, "missing": 84, "extra": 3413, "golden_count": 6865, "ocr_count": 10194, "accuracy_percent": 80.277}, "number_tokens": {"correct": 236, "substituted": 35, "missing": 93, "extra": 16, "accuracy_percent": 64.835}}, "gray220_px1024": {"scored_pages": 21, "words": {"correct": 6614, "substituted": 201, "missing": 50, "extra": 59, "golden_count": 6865, "ocr_count": 6874, "accuracy_percent": 96.344}, "number_tokens": {"correct": 312, "substituted": 43, "missing": 9, "extra": 5, "accuracy_percent": 85.714}}, "raw_px1400": {"scored_pages": 21, "words": {"correct": 5832, "substituted": 650, "missing": 383, "extra": 4286, "golden_count": 6865, "ocr_count": 10768, "accuracy_percent": 84.953}, "number_tokens": {"correct": 272, "substituted": 15, "missing": 77, "extra": 5, "accuracy_percent": 74.725}}, "gray220_px1400": {"scored_pages": 21, "words": {"correct": 6839, "substituted": 21, "missing": 5, "extra": 2, "golde